# Lipschitz SDF — pipeline mesh reel (SLL, avec ou sans encodage parametrique)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import trimesh
import plotly.graph_objects as go
import plotly.colors as pc 
import contextlib, io

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"device = {device}")

ValueError: module functions cannot set METH_CLASS or METH_STATIC

In [2]:
# ===== architecture : SLL 1-Lipschitz, avec encodage parametrique ou sans (parametre `encoding`) =====

class SLLLayer(nn.Module):
    """Semi-definite Programming Lipschitz Layer (Araujo et al.) — residuelle, 1-Lipschitz par construction."""
    def __init__(self, k, scale=1.0):
        super().__init__()
        self.scale = scale
        self.W = nn.Parameter(torch.randn(k, k) * (1.0 / k**0.5))
        self.b = nn.Parameter(torch.zeros(k))
        self.q = nn.Parameter(torch.zeros(k))

    def forward(self, x):
        WtW = self.W.T @ self.W
        q = self.q
        T = torch.abs(WtW * torch.exp(q[None, :] - q[:, None])).sum(dim=1)
        T_inv = 1.0 / (T + 1e-8)
        h = torch.relu(x @ self.W + self.b)
        return (x - 2.0 * (h * T_inv) @ self.W.T) * self.scale


class LossHKR:
    """Hinge-Kantorovitch-Rubinstein loss (Serrurier et al. / Coiffier & Bethune)."""
    def __init__(self, margin, lbda):
        self.margin = margin
        self.lbda = lbda

    def __call__(self, y):
        return F.relu(self.margin - y) + (1.0 / self.lbda) * torch.mean(-y)


class LipschitzParametricEncoding(nn.Module):
    """Grille trilineaire de features, chaque cellule projetee par SVD pour rester 1-Lipschitz
    (sigma_max <= dx/sqrt(4) en 3D)."""
    def __init__(self, resolution=32, feature_dim=8, domain=(-2, 2), constant_majoration=True, project_all=False):
        super().__init__()
        self.resolution = resolution
        self.feature_dim = feature_dim
        self.domain = domain
        self.dx = (domain[1] - domain[0]) / (resolution - 1)
        self.table_size = resolution ** 3
        self.table = nn.Parameter(torch.empty(feature_dim, self.table_size).uniform_(-1e-4, 1e-4))
        self.constant_majoration = constant_majoration
        self.majoration_treshold = self.dx / np.sqrt(4)
        self.project_all = project_all

    def L(self, x):
        indices = self.get_indices(x)
        Aij = self.Aij(indices)
        grad_Dij = self.grad_Dij(x, indices[0])
        Lij = torch.bmm(Aij, grad_Dij)
        return torch.linalg.matrix_norm(Lij, ord=2)

    def grad_Dij(self, x, xijk):
        dx = self.dx
        a = x[:, 0] - xijk[:, 0]; b = x[:, 1] - xijk[:, 1]; c = x[:, 2] - xijk[:, 2]
        ia, ib, ic = dx - a, dx - b, dx - c
        grad = torch.stack([
            torch.stack([-ib*ic, -ia*ic, -ia*ib], dim=-1),
            torch.stack([ ib*ic, -a*ic,  -a*ib ], dim=-1),
            torch.stack([-b*ic,   ia*ic, -ia*b ], dim=-1),
            torch.stack([ b*ic,   a*ic,  -a*b  ], dim=-1),
            torch.stack([-ib*c,  -ia*c,   ia*ib], dim=-1),
            torch.stack([ ib*c,  -a*c,    a*ib ], dim=-1),
            torch.stack([-b*c,    ia*c,   ia*b ], dim=-1),
            torch.stack([ b*c,    a*c,    a*b  ], dim=-1),
        ], dim=1) / (dx**3)
        return grad

    def hash(self, index):
        index = torch.round(self.normalize(index)).long()
        return (index[:, 0] * self.resolution**2 + index[:, 1] * self.resolution + index[:, 2])

    def hash_indices(self, indices):
        return [self.hash(i) for i in indices]

    def normalize(self, x):
        lo, hi = self.domain
        return (x - lo) / (hi - lo) * (self.resolution - 1)

    def get_indices(self, x):
        lo = self.domain[0]
        xr = self.normalize(x)
        i0 = torch.floor(xr).long().clamp(0, self.resolution - 2)
        x000 = lo + i0.float() * self.dx
        d = self.dx
        offsets = torch.tensor([[0,0,0],[d,0,0],[0,d,0],[d,d,0],[0,0,d],[d,0,d],[0,d,d],[d,d,d]], device=x.device)
        return tuple(x000 + off for off in offsets)

    def D(self, x, xijk):
        Dx = (x[:, 0] - xijk[:, 0]) / self.dx; Dy = (x[:, 1] - xijk[:, 1]) / self.dx; Dz = (x[:, 2] - xijk[:, 2]) / self.dx
        return torch.stack([(1-Dx)*(1-Dy)*(1-Dz),(Dx)*(1-Dy)*(1-Dz),(1-Dx)*(Dy)*(1-Dz),(Dx)*(Dy)*(1-Dz),
                             (1-Dx)*(1-Dy)*(Dz),(Dx)*(1-Dy)*(Dz),(1-Dx)*(Dy)*(Dz),(Dx)*(Dy)*(Dz)], dim=-1).unsqueeze(-1)

    def Aij(self, indices):
        corners = [self.table[:, self.hash(i)].T for i in indices]
        return torch.stack(corners, dim=-1)

    def forward(self, x):
        indices = self.get_indices(x)
        Aij = self.Aij(indices)
        Dij = self.D(x, indices[0])
        return torch.bmm(Aij, Dij).squeeze(-1)

    def project_sub_matrix(self, x, treshold):
        indices = self.get_indices(x)
        Aij = self.Aij(indices)
        hashes = self.hash_indices(indices)
        if isinstance(treshold, torch.Tensor) and treshold.dim() > 0:
            treshold = treshold.unsqueeze(-1)
        with torch.no_grad():
            U, S, Vt = torch.linalg.svd(Aij, full_matrices=False)
            S_clipped = torch.clamp(S, max=treshold)
            Aij_projected = U @ torch.diag_embed(S_clipped) @ Vt
            for c in range(8):
                self.table[:, hashes[c]] = Aij_projected[:, :, c].T

    def project_majoration(self, x):
        self.project_sub_matrix(x, self.majoration_treshold)


class LipschitzModel(nn.Module):
    """SLL 1-Lipschitz. encoding=True : entree passee par LipschitzParametricEncoding avant les
    couches SLL. encoding=False : coordonnees brutes zero-paddees vers k, sans grille."""
    def __init__(self, encoding=True, resolution=64, feature_dim=16, k=128, n_layers=20,
                 domain=(-2, 2), add_positions=True, project_all=False, constant_majoration=True,
                 divide=False, sll_divide=True):
        super().__init__()
        self.encoding = encoding
        self.add_positions = add_positions if encoding else False
        self.k = k
        if encoding:
            self.enc = LipschitzParametricEncoding(resolution, feature_dim, domain=domain,
                                                     project_all=project_all, constant_majoration=constant_majoration)
            self.in_dim = feature_dim + (3 if self.add_positions else 0)
        else:
            self.enc = None
            self.in_dim = 3
        last = SLLLayer(k, scale=1.0 / 2**0.5) if (encoding and sll_divide) else SLLLayer(k)
        rest = [SLLLayer(k) for _ in range(n_layers - 1)]
        self.sll = nn.Sequential(*rest, last)
        self.w_out = nn.Parameter(torch.randn(k) / k**0.5)
        self.b_out = nn.Parameter(torch.zeros(1))
        self.divide = divide

    def L_pre(self, x):
        if not self.encoding:
            return torch.ones(x.shape[0], device=x.device)
        return torch.sqrt(self.enc.L(x)**2 + 1) if self.add_positions else self.enc.L(x)

    def forward(self, x):
        if self.encoding:
            feat = self.enc(x)
            if self.add_positions:
                feat = torch.cat([feat, x], dim=-1)
        else:
            feat = x
        feat = F.pad(feat, (0, self.k - self.in_dim))
        feat = self.sll(feat)
        scale = self.w_out.norm() * (2**0.5 if self.divide and self.add_positions else 1.0)
        return (feat @ self.w_out) / scale + self.b_out

In [3]:
# ===== geometrie differentielle : courbure par differences finies + calibration de h =====

def curvature_fd(f, x, h=1e-2):
    f0 = f(x)
    if f0.dim() == 2:
        f_raw = f
        f = lambda p: f_raw(p).squeeze(-1)
        f0 = f0.squeeze(-1)
    hx = torch.zeros_like(x); hx[:, 0] = h
    hy = torch.zeros_like(x); hy[:, 1] = h
    hz = torch.zeros_like(x); hz[:, 2] = h
    dxf = (f(x + hx) - f(x - hx)) / (2*h)
    dyf = (f(x + hy) - f(x - hy)) / (2*h)
    dzf = (f(x + hz) - f(x - hz)) / (2*h)
    dxxf = (f(x + hx) - 2*f0 + f(x - hx)) / h**2
    dyyf = (f(x + hy) - 2*f0 + f(x - hy)) / h**2
    dzzf = (f(x + hz) - 2*f0 + f(x - hz)) / h**2
    dxyf = (f(x+hx+hy) - f(x+hx-hy) - f(x-hx+hy) + f(x-hx-hy)) / (4*h**2)
    dxzf = (f(x+hx+hz) - f(x+hx-hz) - f(x-hx+hz) + f(x-hx-hz)) / (4*h**2)
    dyz  = (f(x+hy+hz) - f(x+hy-hz) - f(x-hy+hz) + f(x-hy-hz)) / (4*h**2)
    grad_norm = torch.sqrt(dxf**2 + dyf**2 + dzf**2)
    curvature = (dxxf*(dyf**2+dzf**2) + dyyf*(dxf**2+dzf**2) + dzzf*(dxf**2+dyf**2)
                 - 2*(dxf*dyf*dxyf + dxf*dzf*dxzf + dyf*dzf*dyz)) / (grad_norm**3)
    return curvature


def calibrate_h_min(model, surface_pts, candidates=(0.01,0.02,0.05,0.1,0.15,0.2,0.3), tol=0.15):
    prev = None
    for h in candidates:
        k = curvature_fd(model, surface_pts, h).abs().mean().item()
        if prev is not None and abs(k - prev) / max(prev, 1e-6) < tol:
            return h
        prev = k
    return candidates[-1]

In [4]:
# ===== octree adaptatif : subdivision par courbure + equilibrage 2:1 =====

def compute_octree(model, domain=(-2,2), max_depth=6, min_depth=1, curvature_threshold=0.1,
                    device='cpu', fd=False, hmin=None):
    half0 = (domain[1] - domain[0]) / 2.0
    center0 = torch.full((3,), (domain[0] + domain[1]) / 2.0, device=device)

    def subdivide(center, half):
        offsets = torch.tensor([[-1,-1,-1],[-1,-1,1],[-1,1,-1],[-1,1,1],
                                 [1,-1,-1],[1,-1,1],[1,1,-1],[1,1,1]], dtype=center.dtype, device=center.device)
        return [center + offsets[i]*(half/2.0) for i in range(8)]

    def recurse(center, half, depth):
        x = center.unsqueeze(0)
        if fd:
            hc = half*0.1
            h = max(hc, hmin) if hmin is not None else hc
            kappa = curvature_fd(model, x, h=h).abs().item()
        else:
            kappa = 0.0
        should_subdivide = (depth < min_depth) or (((kappa*half)) > curvature_threshold and depth < max_depth)
        if should_subdivide:
            leaves = []
            for cc in subdivide(center, half):
                leaves += recurse(cc, half/2.0, depth+1)
            return leaves
        return [(center, half, depth)]

    return recurse(center0, half0, 0)


def to_grid_coords(center, depth, domain, half0):
    side = 2*half0/2**depth
    return tuple(int(round((center[k].item()-domain[0])/side - 0.5)) for k in range(3))


def balance_octree(leaves, domain=(-2,2), max_passes=10):
    """Deux feuilles adjacentes par une face ne different jamais de plus d'un niveau de profondeur."""
    half0 = (domain[1]-domain[0])/2.0
    offsets = torch.tensor([[-1,-1,-1],[-1,-1,1],[-1,1,-1],[-1,1,1],
                             [1,-1,-1],[1,-1,1],[1,1,-1],[1,1,1]], dtype=torch.float32)
    leaves = list(leaves)
    for _ in range(max_passes):
        leaf_at = {}
        for i,(c,h,d) in enumerate(leaves):
            leaf_at[(d, *to_grid_coords(c,d,domain,half0))] = i
        to_split = set()
        for i,(c,h,d) in enumerate(leaves):
            gx,gy,gz = to_grid_coords(c,d,domain,half0)
            n_cells = 2**d
            for axis in range(3):
                for sign in (-1,1):
                    ng=[gx,gy,gz]; ng[axis]+=sign
                    if not (0<=ng[axis]<n_cells): continue
                    for dd in range(d-1,-1,-1):
                        shift=d-dd
                        coarse=tuple(v>>shift for v in ng)
                        key=(dd,*coarse)
                        if key in leaf_at:
                            if dd < d-1: to_split.add(leaf_at[key])
                            break
        if not to_split: return leaves
        new_leaves=[]
        for i,(c,h,d) in enumerate(leaves):
            if i in to_split:
                for off in offsets:
                    new_leaves.append((c+off.to(c.device)*(h/2.0), h/2.0, d+1))
            else:
                new_leaves.append((c,h,d))
        leaves = new_leaves
    return leaves

In [5]:
# ===== extraction de surface : dual contouring sur l'octree equilibre =====

CORNER_OFF = torch.tensor([[sx,sy,sz] for sx in (-1,1) for sy in (-1,1) for sz in (-1,1)], dtype=torch.float32)
EDGES = [(0,1),(1,3),(3,2),(2,0),(4,5),(5,7),(7,6),(6,4),(0,4),(1,5),(2,6),(3,7)]

def grad_fd(f, x, h=1e-3):
    g = torch.zeros_like(x)
    for i in range(x.shape[1]):
        e = torch.zeros_like(x); e[:,i]=h
        g[:,i] = (f(x+e)-f(x-e))/(2*h)
    return g


def dual_contouring(model, leaves, domain=(-2,2), device='cpu', newton_iters=3):
    """PREREQUIS : leaves passe par balance_octree. Retourne (verts, faces triangles)."""
    half0 = (domain[1]-domain[0])/2.0
    dmax = max(d for _,_,d in leaves)
    min_half = min(float(h) for _,h,_ in leaves)
    q = min_half/64.0
    def key(p): return tuple(torch.round(p/q).to(torch.int64).tolist())

    def to_grid(c,d):
        side = 2*half0/2**d
        return tuple(int(torch.floor((c[k]-domain[0])/side).item()) for k in range(3))
    leaf_at = {(d,*to_grid(c,d)): i for i,(c,h,d) in enumerate(leaves)}

    def locate(p):
        for d in range(dmax,-1,-1):
            side = 2*half0/2**d
            g = tuple(int(torch.floor((p[k]-domain[0])/side).item()) for k in range(3))
            if any(v<0 or v>=2**d for v in g): return None
            li = leaf_at.get((d,*g))
            if li is not None: return li
        return None

    corner_off = CORNER_OFF.to(device)
    sign_change, corner_f = [], {}
    for li,(c,h,d) in enumerate(leaves):
        corners = c.to(device) + corner_off*float(h)
        with torch.no_grad():
            fc = model(corners).reshape(-1)
        corner_f[li] = fc
        s = torch.where(fc>=0,1,-1)
        if s.min() < s.max(): sign_change.append(li)

    if not sign_change:
        raise RuntimeError("aucune cellule ne change de signe : reseau sous-entraine ou octree jamais "
                            "subdivise (verifier min_depth / curvature_threshold / nb_epochs)")

    minimizers = {}
    def get_min(li):
        if li not in minimizers:
            c,h,d = leaves[li]
            m = c.to(device).clone().unsqueeze(0)
            with torch.no_grad():
                for _ in range(newton_iters):
                    g = grad_fd(model, m)
                    m = m - model(m).reshape(-1,1)*g/(g**2).sum(-1,keepdim=True).clamp_min(1e-12).sqrt()
            m = torch.max(torch.min(m[0], c.to(device)+float(h)), c.to(device)-float(h))
            minimizers[li]=m
        return minimizers[li]

    eps = min_half*0.5
    faces, seen = [], set()
    li2row = {}
    for li in sign_change:
        c,h,d = leaves[li]; c = c.to(device)
        corners = c + corner_off*float(h)
        fc = corner_f[li]
        for a,b in EDGES:
            if (fc[a]>=0)==(fc[b]>=0): continue
            p1,p2 = corners[a], corners[b]
            axis = int(torch.argmax((p2-p1).abs()).item())
            mid = (p1+p2)/2
            perp = [k for k in range(3) if k!=axis]
            adj=[]
            for s1 in (-1,1):
                for s2 in (-1,1):
                    pp = mid.clone(); pp[perp[0]]+=s1*eps; pp[perp[1]]+=s2*eps
                    lj = locate(pp)
                    if lj is not None: adj.append(lj)
            if any(leaves[lj][2]>d for lj in set(adj)): continue
            uniq = list(dict.fromkeys(adj))
            if len(uniq)<3: continue
            ck=(key(mid),axis)
            if ck in seen: continue
            seen.add(ck)
            mpts = torch.stack([get_min(lj) for lj in uniq])
            ax = (p2-p1)/(p2-p1).norm()
            ctr = mpts.mean(0)
            u = mpts[0]-ctr; u = u-(u@ax)*ax; u=u/u.norm().clamp_min(1e-12)
            v = torch.cross(ax,u,dim=0)
            ang = torch.atan2((mpts-ctr)@v,(mpts-ctr)@u)
            order = [uniq[i] for i in ang.argsort().tolist()]
            pq = torch.stack([get_min(lj) for lj in order])
            with torch.no_grad():
                outward = grad_fd(model, ctr.unsqueeze(0))[0]
            if torch.cross(pq[1]-pq[0], pq[2]-pq[0], dim=0)@outward < 0: order = order[::-1]
            for lj in order: li2row.setdefault(lj, len(li2row))
            rows = [li2row[lj] for lj in order]
            faces.append((rows[0],rows[1],rows[2]))
            if len(rows)==4: faces.append((rows[0],rows[2],rows[3]))

    inv = {v:k for k,v in li2row.items()}
    verts = torch.stack([minimizers[inv[i]] for i in range(len(li2row))])
    return verts.cpu(), torch.tensor(faces)

In [6]:
# ===== visualisation : octree, mesh plein, wireframe, courbure =====

def plot_octree(leaves):
    edges = [(0,1),(1,3),(3,2),(2,0),(4,5),(5,7),(7,6),(6,4),(0,4),(1,5),(2,6),(3,7)]
    corner_offsets = np.array([[-1,-1,-1],[-1,-1,1],[-1,1,-1],[-1,1,1],
                                [1,-1,-1],[1,-1,1],[1,1,-1],[1,1,1]])
    depths = sorted(set(d for _, _, d in leaves))
    palette = pc.sample_colorscale('Viridis', [i / max(len(depths) - 1, 1) for i in range(len(depths))])
    fig = go.Figure()
    for depth, color in zip(depths, palette):
        xs, ys, zs = [], [], []
        for center, half, d in leaves:
            if d != depth: continue
            c = center.detach().cpu().numpy() if torch.is_tensor(center) else np.asarray(center)
            corners = c + corner_offsets * float(half)
            for i, j in edges:
                xs += [corners[i,0], corners[j,0], None]
                ys += [corners[i,1], corners[j,1], None]
                zs += [corners[i,2], corners[j,2], None]
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                                    line=dict(color=color, width=2), name=f'profondeur {depth}'))
    fig.update_layout(scene=dict(aspectmode='data'),
                       title=f'Octree — {len(leaves)} feuilles, {len(depths)} niveaux',
                       margin=dict(l=0, r=0, t=40, b=0))
    fig.show()


def plot_mesh(verts, faces, domain=(-2,2)):
    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        color='lightblue', opacity=1.0, flatshading=True,
    )])
    fig.update_layout(scene=dict(aspectmode='cube',
                                  xaxis=dict(range=list(domain)), yaxis=dict(range=list(domain)),
                                  zaxis=dict(range=list(domain))))
    fig.show()


def plot_mesh_wireframe(verts, faces, domain=(-2,2)):
    edge_x, edge_y, edge_z, edge_c, lengths = [], [], [], [], []
    for tri in faces:
        for a, b in [(0,1),(1,2),(2,0)]:
            i, j = tri[a], tri[b]
            length = (verts[i]-verts[j]).norm().item()
            lengths.append(length)
            edge_x.extend([verts[i,0], verts[j,0], None])
            edge_y.extend([verts[i,1], verts[j,1], None])
            edge_z.extend([verts[i,2], verts[j,2], None])
            edge_c.extend([length, length, float('nan')])
    lengths = torch.tensor(lengths)
    cmax = lengths.quantile(0.95).item()
    fig = go.Figure(data=[go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z, mode='lines',
        line=dict(color=edge_c, colorscale='Bluered_r', cmin=lengths.min().item(), cmax=cmax,
                  width=3, showscale=True, colorbar=dict(title='longueur arete')),
        hoverinfo='skip')])
    fig.update_layout(title="Wireframe (rouge=fin, bleu=grossier)",
                       scene=dict(aspectmode='cube', xaxis=dict(range=list(domain)),
                                  yaxis=dict(range=list(domain)), zaxis=dict(range=list(domain))),
                       margin=dict(l=0, r=0, t=40, b=0))
    fig.show()


def plot_curvature_on_mesh(model, verts, faces, h_min, domain=(-2,2)):
    with torch.no_grad():
        kappa = curvature_fd(model, verts.to(device), h_min).abs()
    cmax = kappa.quantile(0.95).item()
    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=kappa.cpu().numpy(), colorscale='Bluered', cmin=0, cmax=cmax,
        showscale=True, colorbar=dict(title='courbure'))])
    fig.update_layout(title='Courbure sur la surface',
                       scene=dict(aspectmode='cube', xaxis=dict(range=list(domain)),
                                  yaxis=dict(range=list(domain)), zaxis=dict(range=list(domain))),
                       margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

In [ ]:
def load_mesh_pools(mesh_path, domain=(-2, 2), n_occ=2_000_000):
    mesh = trimesh.load(mesh_path)
    v = mesh.vertices - mesh.vertices.mean(axis=0)
    v = v / np.linalg.norm(v, axis=1).max()
    mesh.vertices = v
    print(f"  watertight={mesh.is_watertight}")

    pts_np = np.random.uniform(domain[0], domain[1], size=(n_occ, 3)).astype(np.float32)
    inside = mesh.contains(pts_np)
    print(f"  occupation: {inside.mean()*100:.2f}% dedans")
    points_pool = torch.from_numpy(pts_np)
    y_pool = torch.where(torch.from_numpy(inside), -1.0, 1.0)

    return mesh, points_pool, y_pool

In [8]:
# ===== entrainement hKR (tirage equilibre dans le pool d'occupation precalcule) =====

def _default_lambda_schedule(nb_epochs):
    fractions = [0, 0.0625, 0.125, 0.25, 0.375, 0.5, 0.625]
    values    = [10, 30, 60, 100, 200, 400, 800]
    return {int(f * nb_epochs): v for f, v in zip(fractions, values)}


def train_Lipschitz_generic(model, points_pool, y_pool, nb_samples, lr, nb_epochs,
                             m=0.01, lambda_schedule=None, parallel=False, encoding=True,
                             majoration=False, log_every=None):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if parallel and torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for training.")
        model = nn.DataParallel(model)
    model = model.to(device)
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    enc = raw_model.enc if encoding else None
    majoration = majoration and (enc is not None)  # ignore silencieusement si pas d'encodage

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if lambda_schedule is None:
        lambda_schedule = {100*i: max(min(10*i, 100), 1) for i in range(nb_epochs // 100)}
    loss_fn = LossHKR(margin=m, lbda=1.0)
    history = []
    log_every = log_every or max(nb_epochs // 10, 1)

    points_pool, y_pool = points_pool.to(device), y_pool.to(device)
    idx_in  = (y_pool == -1).nonzero(as_tuple=True)[0]
    idx_out = (y_pool == +1).nonzero(as_tuple=True)[0]
    N = points_pool.shape[0]
    print(f"  pool: {len(idx_in)} dedans / {len(idx_out)} dehors ({len(idx_in)/N*100:.2f}% dedans)")

    for epoch in range(nb_epochs):
        if epoch in lambda_schedule:
            loss_fn.lbda = lambda_schedule[epoch]

        half = nb_samples // 2
        i_in  = idx_in[torch.randint(0, len(idx_in), (half,), device=device)]
        i_out = idx_out[torch.randint(0, len(idx_out), (nb_samples - half,), device=device)]
        idx = torch.cat([i_in, i_out])
        x, y = points_pool[idx], y_pool[idx]

        pred = model(x).squeeze()
        loss = loss_fn(y * pred).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if majoration:
            enc.project_majoration(x)

        history.append(loss.item())
        if epoch % log_every == 0:
            if encoding:
                with torch.no_grad():
                    pre_max_L = raw_model.L_pre(x).max().item()
                    total_L = pre_max_L / (2**0.5) if raw_model.add_positions else pre_max_L
                print(f"  epoch {epoch:5d} | loss={loss.item():.5f} | L_total~{total_L:.4f}")
            else:
                print(f"  epoch {epoch:5d} | loss={loss.item():.5f}")

    return model, history

In [ ]:
# ===== pipeline complet (1 objet), et wrapper batch (plusieurs objets, erreurs isolees) =====

def run_lipschitz_pipeline(name, mesh_path, domain=(-2, 2), encoding=True,
                            resolution=64, feature_dim=16, k=128, n_layers=20,
                            add_positions=True, project_all=False, constant_majoration=True,
                            divide=False, sll_divide=True,
                            nb_samples=100_000, lr=1e-3, nb_epochs=8000, m=0.01,
                            lambda_schedule=None, parallel=True, majoration=True,
                            n_occ=2_000_000,
                            max_depth=6, min_depth=1, curvature_threshold=1.2,
                            plot=True, device=None, verbose_octree=False):
    device = device or torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    globals()['device'] = device  # plot_curvature_on_mesh lit la globale, pas un parametre

    print(f"[{name}] chargement + pools...")
    mesh, points_pool, y_pool = load_mesh_pools(mesh_path, domain, n_occ)

    print(f"[{name}] entrainement (encoding={encoding})...")
    model, history = train_Lipschitz_generic(
        LipschitzModel(encoding=encoding, resolution=resolution, feature_dim=feature_dim, k=k,
                        n_layers=n_layers, domain=domain, add_positions=add_positions,
                        project_all=project_all, constant_majoration=constant_majoration,
                        divide=divide, sll_divide=sll_divide),
        points_pool, y_pool, nb_samples=nb_samples, lr=lr, nb_epochs=nb_epochs,
        m=m, lambda_schedule=lambda_schedule or _default_lambda_schedule(nb_epochs),
        parallel=parallel, encoding=encoding, majoration=majoration,
    )
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.eval()

    probe = torch.rand(4000, 3, device=device) * (domain[1] - domain[0]) + domain[0]
    with torch.no_grad():
        dists = raw_model(probe).reshape(-1).abs()
    near_surface = probe[dists < 0.1][:200]
    if len(near_surface) == 0:
        print(f"[{name}] aucun point sous le seuil 0.1 — repli sur les 200 plus proches")
        near_surface = probe[dists.argsort()[:200]]
    h_min = calibrate_h_min(raw_model, near_surface)
    print(f"[{name}] h_min={h_min}")

    def _build_octree():
        return compute_octree(raw_model, domain=domain, max_depth=max_depth, min_depth=min_depth,
                               curvature_threshold=curvature_threshold, device=device, fd=True, hmin=h_min)
    if verbose_octree:
        leaves = _build_octree()
    else:
        with contextlib.redirect_stdout(io.StringIO()):
            leaves = _build_octree()
    print(f"[{name}] {len(leaves)} feuilles générées")
    if plot:
        plot_octree(leaves)
    balanced_leaves = balance_octree(leaves, domain=domain, max_passes=10)

    verts, faces = dual_contouring(raw_model, balanced_leaves, domain=domain, device=device, newton_iters=3)
    print(f"[{name}] mesh extrait : {verts.shape[0]} sommets, {faces.shape[0]} faces")

    if plot:
        plot_mesh(verts, faces, domain=domain)
        plot_mesh_wireframe(verts, faces, domain=domain)
        plot_curvature_on_mesh(raw_model, verts, faces, h_min, domain=domain)

    return {"name": name, "model": raw_model, "history": history, "verts": verts, "faces": faces,
            "leaves": leaves, "balanced_leaves": balanced_leaves, "h_min": h_min, "mesh": mesh}


def run_lipschitz_pipeline_batch(specs, stop_on_error=False, **shared_kwargs):
    """specs : liste de dicts, chacun avec au moins 'name' + 'mesh_path'.
    shared_kwargs : parametres communs, ecrasables par spec. Un objet qui echoue
    n'interrompt pas les suivants (capture sous results[name]['error'])."""
    results = {}
    for spec in specs:
        cfg = {**shared_kwargs, **spec}
        name = cfg.pop("name")
        print(f"\n{'='*70}\n  {name}\n{'='*70}")
        try:
            results[name] = run_lipschitz_pipeline(name=name, **cfg)
        except Exception as e:
            print(f"[{name}] ECHEC : {type(e).__name__}: {e}")
            if stop_on_error:
                raise
            results[name] = {"name": name, "error": str(e)}
    return results

: 

In [ ]:
batch = run_lipschitz_pipeline_batch([
    {"name": "teapot_noenc",   "mesh_path": "./models/teapot.obj",   "encoding": False},
    {"name": "teapot_enc",   "mesh_path": "./models/teapot.obj",   "encoding": True},
], plot=True, domain=(-1.2,1.2))

for name, r in batch.items():
    if "error" in r:
        print(f"{name}: ECHEC — {r['error']}")
    else:
        print(f"{name}: {r['verts'].shape[0]} sommets, {r['faces'].shape[0]} faces")


  teapot_noenc
[teapot_noenc] chargement + pools...
  watertight=False


In [ ]:
for name, r in batch.items():
    if "error" in r:
        print(f"{name}: ECHEC — {r['error']}")
        continue
    print(f"=== {name} ===")
    plot_octree(r["leaves"])
    plot_mesh(r["verts"], r["faces"])
    plot_mesh_wireframe(r["verts"], r["faces"])
    plot_curvature_on_mesh(r["model"], r["verts"], r["faces"], r["h_min"])

=== suzanne_noenc ===


=== suzanne_enc ===


=== cow_noenc ===


=== cow_enc ===
